# XRPL AML — corrected Behaviour + LightGBM submission

**Fixes:** build `test_features` before merging; train and submit with the same 18 features; match all rows by `TRANSACTION_HASH`; isolate model variables; verify predictions. Run **Kernel → Restart & Run All**. Notebook expected in `notebooks/` under the repository.

**Important:** This fixes a verifiable submission mismatch. It does not guarantee higher hidden-test AP. Requires the organiser `payments.py`, train/test data, and train/test behaviour parquet files.

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier


import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/payments.py').is_file() and (p / 'notebooks').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Run from the project root or its notebooks directory.')
sys.path.insert(0, str(ROOT))
from src.payments import load_payments
TRAIN_DIR = ROOT / 'data/raw/train'
TEST_DIR = ROOT / 'data/raw/test'
OUTPUT_DIR = ROOT / 'outputs' 

B_PARAMS = dict(
    objective='binary', n_estimators=300, learning_rate=0.03,
    num_leaves=15, max_depth=5, min_child_samples=100,
    colsample_bytree=0.8, reg_lambda=1.0, class_weight=None,
    random_state=0, verbosity=-1, n_jobs=-1, subsample=0.8, subsample_freq=1
)

for path in [TRAIN_DIR / 'transactions.parquet', TEST_DIR / 'transactions.parquet',
             OUTPUT_DIR / 'train_behaviour_features.parquet',
             OUTPUT_DIR / 'test_behaviour_features.parquet']:
    assert path.exists(), f'Missing file: {path}'
print('Input files found')

Input files found


In [11]:
HOUR, TEN_MINUTES = 3600, 600


def count_in_window(times_by_account, account, t, before, after):
    """How many of an account's payments fall between t - before and t + after."""
    times = times_by_account.get(account)
    if times is None:
        return 0
    return int(np.searchsorted(times, t + after, "right") - np.searchsorted(times, t - before, "left"))


def make_features(df):
    df = df.sort_values("TIMESTAMP").reset_index(drop=True)
    t = df.TIMESTAMP.astype(int).to_numpy()
    value = df.VALUE.to_numpy()
    sends = {a: g.TIMESTAMP.astype(int).to_numpy() for a, g in df.groupby("FROM_ADDRESS")}
    gets = {a: g.TIMESTAMP.astype(int).to_numpy() for a, g in df.groupby("TO_ADDRESS")}

    f = pd.DataFrame({"TRANSACTION_HASH": df.TRANSACTION_HASH})

    # Rule: bigger payments are more interesting (log keeps huge ones in range).
    f["log_value"] = np.log1p(value)

    # Rule: how busy are the sender and the receiver over the whole week?
    f["sender_payments"] = df.FROM_ADDRESS.map(df.FROM_ADDRESS.value_counts())
    f["receiver_payments"] = df.TO_ADDRESS.map(df.TO_ADDRESS.value_counts())

    # Rule: the receiver sends money on within an hour (a "relay" or "mule").
    f["receiver_forwards_1h"] = [
        count_in_window(sends, a, ti, 0, HOUR) > 0 for a, ti in zip(df.TO_ADDRESS, t)
    ]

    # Rule: the sender only just received money (it is passing it along).
    f["sender_received_1h_before"] = [
        count_in_window(gets, a, ti, HOUR, 0) > 0 for a, ti in zip(df.FROM_ADDRESS, t)
    ]

    # Rule: fan-out / fan-in, many payments from one sender or into one
    # receiver within ten minutes.
    f["sender_burst_10m"] = [
        count_in_window(sends, a, ti, TEN_MINUTES, TEN_MINUTES) for a, ti in zip(df.FROM_ADDRESS, t)
    ]
    f["receiver_burst_10m"] = [
        count_in_window(gets, a, ti, TEN_MINUTES, TEN_MINUTES) for a, ti in zip(df.TO_ADDRESS, t)
    ]

    # Rule: the receiver passes on a similar amount (within 10%) within an hour,
    # a classic sign of money being layered through an account.
    sent_values = {a: g.VALUE.to_numpy() for a, g in df.groupby("FROM_ADDRESS")}

    def forwards_similar(account, ti, amount):
        times = sends.get(account)
        if times is None:
            return False
        lo, hi = np.searchsorted(times, ti, "right"), np.searchsorted(times, ti + HOUR, "right")
        later = sent_values[account][lo:hi]
        return bool(np.any(np.abs(later - amount) <= 0.1 * amount))

    f["receiver_forwards_similar_1h"] = [
        forwards_similar(a, ti, v) for a, ti, v in zip(df.TO_ADDRESS, t, value)
    ]

    # Rule: the sender received a similar amount within the hour before.
    got_values = {a: g.VALUE.to_numpy() for a, g in df.groupby("TO_ADDRESS")}

    def received_similar(account, ti, amount):
        times = gets.get(account)
        if times is None:
            return False
        lo, hi = np.searchsorted(times, ti - HOUR, "left"), np.searchsorted(times, ti, "left")
        earlier = got_values[account][lo:hi]
        return bool(np.any(np.abs(earlier - amount) <= 0.1 * amount))

    f["sender_received_similar_1h"] = [
        received_similar(a, ti, v) for a, ti, v in zip(df.FROM_ADDRESS, t, value)
    ]

    # Rule: how quickly does the receiver send money on? (minutes, capped at 60)
    def minutes_to_next_send(account, ti):
        times = sends.get(account)
        if times is None:
            return 60.0
        i = np.searchsorted(times, ti, "right")
        return min(60.0, (times[i] - ti) / 60) if i < len(times) else 60.0

    f["receiver_minutes_to_forward"] = [
        minutes_to_next_send(a, ti) for a, ti in zip(df.TO_ADDRESS, t)
    ]

    # Rule: pass-through accounts send on most of what they receive over the week.
    sent_total = df.groupby("FROM_ADDRESS").VALUE.sum()
    got_total = df.groupby("TO_ADDRESS").VALUE.sum()
    pass_through = (sent_total / got_total.reindex(sent_total.index)).clip(0, 3)
    f["sender_pass_through"] = df.FROM_ADDRESS.map(pass_through).fillna(0)
    f["receiver_pass_through"] = df.TO_ADDRESS.map(pass_through).fillna(0)

    # ---- Add your own rules here, for example: ----
    # f["round_amount"] = (value % 100 < 1)
    # f["night_time"] = (t % 86400) < 6 * 3600

    return f.astype({c: float for c in f.columns if c != "TRANSACTION_HASH"})




In [12]:
# Recreate all source features from scratch — no dependence on stale notebook variables.
train = pd.DataFrame(load_payments(TRAIN_DIR))
test = pd.DataFrame(load_payments(TEST_DIR))
train_features = make_features(train)
test_features = make_features(test)

behaviour_cols = [
    'sender_tx_count_1h', 'sender_tx_count_24h',
    'seconds_since_previous_tx', 'sender_unique_recipients_24h',
    'sender_outflow_usd_24h', 'sender_inflow_usd_24h'
]
behaviour_train = pd.read_parquet(OUTPUT_DIR / 'train_behaviour_features.parquet')
behaviour_test = pd.read_parquet(OUTPUT_DIR / 'test_behaviour_features.parquet')

def join_behaviour(base, extra, name):
    key = 'TRANSACTION_HASH'
    required = [key] + behaviour_cols
    assert all(c in extra.columns for c in required), f'{name}: missing columns'
    assert base[key].is_unique and extra[key].is_unique, f'{name}: duplicate transaction hashes'
    assert len(base) == len(extra), f'{name}: row count mismatch'
    assert set(base[key]) == set(extra[key]), f'{name}: transaction hash mismatch'
    assert not (set(behaviour_cols) & set(base.columns)), f'{name}: duplicated feature columns'
    joined = base.merge(extra[required], on=key, how='left', validate='one_to_one', sort=False)
    assert len(joined) == len(base)
    assert joined[key].equals(base[key]), f'{name}: row order changed'
    return joined

combined_train = join_behaviour(train_features, behaviour_train, 'train')
combined_test = join_behaviour(test_features, behaviour_test, 'test')

labels = pd.read_csv(TRAIN_DIR / 'labels.csv', usecols=['TRANSACTION_HASH','is_suspicious'])
labelled = combined_train.merge(labels, on='TRANSACTION_HASH', how='left', validate='one_to_one', sort=False)
assert labelled.is_suspicious.notna().all()
feature_cols = [c for c in combined_train.columns if c != 'TRANSACTION_HASH']
assert feature_cols == [c for c in combined_test.columns if c != 'TRANSACTION_HASH']

def model_inputs(frame):
    X = frame[feature_cols].copy()
    for c in feature_cols:
        X[c] = pd.to_numeric(X[c], errors='raise')
    return X.replace([np.inf,-np.inf],np.nan)

X_all = model_inputs(labelled)
X_test = model_inputs(combined_test)
y_all = labelled.is_suspicious.astype(int).to_numpy()
assert len(X_all)==40131 and len(X_test)==40101
print('Train/Test features:', X_all.shape, X_test.shape)
print('Missing feature values (train):\n', X_all.isna().sum()[lambda x:x>0].to_string())
print('Missing feature values (test):\n', X_test.isna().sum()[lambda x:x>0].to_string())
print('Features:', feature_cols)

Train/Test features: (40131, 18) (40101, 18)
Missing feature values (train):
 seconds_since_previous_tx    678
Missing feature values (test):
 seconds_since_previous_tx    676
Features: ['log_value', 'sender_payments', 'receiver_payments', 'receiver_forwards_1h', 'sender_received_1h_before', 'sender_burst_10m', 'receiver_burst_10m', 'receiver_forwards_similar_1h', 'sender_received_similar_1h', 'receiver_minutes_to_forward', 'sender_pass_through', 'receiver_pass_through', 'sender_tx_count_1h', 'sender_tx_count_24h', 'seconds_since_previous_tx', 'sender_unique_recipients_24h', 'sender_outflow_usd_24h', 'sender_inflow_usd_24h']


## Holdout check (same split): 12 original vs 18 behaviour-augmented features

This comparison is informational, **not** a guarantee of hidden-test AP. The full 7-day graph feature generation is allowed for this offline task; held-out transactions can share wallets with fit transactions.

In [13]:
idx_fit, idx_check = train_test_split(
    np.arange(len(y_all)), test_size=0.2, stratify=y_all, random_state=0
)

base_cols = [c for c in feature_cols if c not in behaviour_cols]
for name, cols in [('Baseline 12', base_cols), ('With Behaviour 18', feature_cols)]:
    candidate = LGBMClassifier(**B_PARAMS)
    candidate.fit(X_all.iloc[idx_fit][cols], y_all[idx_fit])
    scores_tr = candidate.predict_proba(X_all.iloc[idx_fit][cols])[:,1]
    scores_check = candidate.predict_proba(X_all.iloc[idx_check][cols])[:,1]
    print(f'{name}: Train AP={100*average_precision_score(y_all[idx_fit], scores_tr):.2f}, '
          f'Validation AP={100*average_precision_score(y_all[idx_check], scores_check):.2f}')

Baseline 12: Train AP=83.55, Validation AP=50.66
With Behaviour 18: Train AP=91.31, Validation AP=54.44


## Submission prediction — **uses all 18 features**

Fit a fresh final model on all labelled train transactions and predict exactly the combined test rows. No reuse of the holdout/CV `model` variable.

In [14]:
final_model = LGBMClassifier(**B_PARAMS)
final_model.fit(X_all[feature_cols], y_all)
final_scores = final_model.predict_proba(X_test[feature_cols])[:, 1]

predictions = pd.DataFrame({
    'TRANSACTION_HASH': combined_test['TRANSACTION_HASH'].to_numpy(),
    'score': final_scores
})

assert list(predictions.columns)==['TRANSACTION_HASH','score']
assert len(predictions)==40101
assert predictions.TRANSACTION_HASH.is_unique
assert set(predictions.TRANSACTION_HASH)==set(test.TRANSACTION_HASH)
assert predictions.score.notna().all()
assert np.isfinite(predictions.score).all()
assert predictions.score.between(0,1).all()

submission_path = OUTPUT_DIR / 'predictions.csv'
submission_path.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(submission_path,index=False)
print('Saved:',submission_path.resolve())
print('Total features used:',final_model.n_features_in_)
print('Predictions:',len(predictions))
display(predictions.head())

Saved: /Users/leeahreum/Desktop/Data Science msc/bathbytes-organized/outputs/predictions.csv
Total features used: 18
Predictions: 40101


,TRANSACTION_HASH,score
0,A579B06B1E4D944F222CFF58685A291914813DD5534F1E...,0.019301
1,F2528D0230F3FF6FA91659921137966BEE9BCB2EC9ABF2...,0.009186
2,902B458E9A4109D2A78511303E8291E1BD891E1311C5EA...,0.000344
3,588EFFC442E8C0691F567AA438AAEBFD5AEFDA2FD06BBF...,0.000458
4,5BAE67314EC25767EDA51E515287123B9922F8263EFCD4...,0.000719
